# INFO 264, Group Exam 2025
### Kandidate numbers: 0000, 0000, 0000, 000
---
#### Important and relevant properties about the data
#### Your preprocessing steps. For example: your process of feature selection and it's results, your choices when it comes to dimension reduction (why/why not/which method/why that method) etc.
#### Based on what you have learnt from the data, why do you think that your models are best-suited for the task
#### Why the particular parameters of a model that you use, work best
#### How you control over- and underfitting
#### Your choice of evaluaon methods. Which metrics did you choose and why? Addionally, you need to give an explanation based on your intuition about why given methods perform better or worse on the given task. 
#### Finally, as a concluding comment in the Jupyter notebook, you need to write a summary of your results, and discuss consequences of such results. 

## The data
---
* Hotel_Address -  Not relevant for sentiment labelling. Can be used if we want to base the analysis on different hotels
* Additional_Number_of_Scoring -  Not relevant for sentiment labelling. We dont really know what this scoring is
* Review_Date -  Not relevant for sentiment labelling. Can be used if we want to analyse reviews over time
* Average_Score -  Not relevant for sentiment labelling. Can be used if we want to compare the avarage score of hotels. Combined with Review_Date it can be used to analyse avarage score over time.
* Hotel_Name -  Not relevant for sentiment labelling
* Reviewer_Nationality -  Not relevant for sentiment labelling. Can be used if we want to analyse reviews based on where the reviewer is from
* Negative_Review - Relevant for sentiment labelling. A must use for our purpose
* Review_Total_Negative_Word_Counts - Not really relevant for sentiment labelling
* Total_Number_of_Reviews -  Not relevant for sentiment labelling. Could be used to compare hotels
* Positive_Review - Relevant for sentiment labelling. A must use for our purpose
* Review_Total_Positive_Word_Counts -  Not relevant for sentiment labelling.
* Total_Number_of_Reviews_Reviewer_Has_Given -  Not relevant for sentiment labelling.
* Reviewer_Score - Relevant for sentiment labelling. A must use for our purpose
* Tags -  Not relevant for sentiment labelling.
* Days_since_review -  Not relevant for sentiment labelling.
* lat -  Not relevant for sentiment labelling.
* lng -  Not relevant for sentiment labelling.
 

## Preprocessing steps
---
* Remove unnecessary columns from the dataset
    * Takes in the whole dataset and creates a new dataset with the columns that we need.
* Removing special characters
    * Special characters don't really add any semantic meaning to the text, and they can also increase the vocabulary, because words like "room!" and "room?" will be treated as two different words. We should mention that some special characters like emojis, can carry sentiment meaning. Another example is special characters used in urls or email addresses. This can lead to some unreliable results, but these types of characters are not pressent in our dataset to the point where it would actually affect the results
* Removing single characters
    * Single characters can be typos, don't carry sentiment meaning, and will increase the vocabulary.
* Convert multiple spaces to one space
    * Having consistency with spacing, is important for tokenization
* Removing prefixes
    * Much like special characters, prefixes can increase the vocabulary, but it's important to remember that changing the prefix can also change the meaning of words.
* Convert to lowercase
    * Just as some of the other preprocessing steps, converting to lowercase letters will reduce the vocabulary.
* Lemmatizing words
    * Lemmatizing reduces words to their base, dictionary form. This ensures that words with the similar meaning is treated the same, and it also help the model to better understand the underlying meaning of the words. Lemming will always produce valid words, and consider context and meaning. This is opposed to Stemming, that might produce words that have no meaning, and it does not look at context. It is important to note that lemmatization comes at a cost compared to Stemming, but the increased cost is generally worth it compared to Stemming.
* Removing stopwords
    * Removing words that has no significant sentiment meaning. This also helps with reducing the vocabulary size. Important to note that the standard stopwords package contains words with sentiment meaning. This is something we were wary of, and so we could have added some of those words back.


In [7]:
import pandas as pd
def csvConverzzion(_Path, _sPathNew, _useCols = ["Negative_Review", "Positive_Review", "Reviewer_Score"], _sEncoding = "utf-8"):
    _dataset = pd.read_csv(_Path, usecols = _useCols)
    _mergedReviews = []
    for i in _dataset.to_numpy():
        _mergedReviews.append([ i[2], i[0], i[1] ])
    pd.DataFrame(_mergedReviews, columns = ["Reviewer_Score", "Negative_Review", "Positive_Review"]).to_csv(_sPathNew, encoding = _sEncoding)

csvConverzzion('./data/Hotel_Reviews.csv', "./data/Hotel_Reviews_Reduced.csv")

In [ ]:
# Importing the default libraries and functions for our models.
import nltk
nltk.download('wordnet')
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))


import re
from nltk.stem import WordNetLemmatizer

stemmer = WordNetLemmatizer()

# An indenpendent function that cleans the sentences up.
# Returns a list of the cleansed sentences, and the data variable which was used.
def clean_sentences(data):
    all_text = []
    for sen in range(0, len(data)):
        # Remove all the special characters
        text = re.sub(r'\W', ' ', str(data.iloc[sen]))
        # remove all single characters
        text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
        # Remove single characters from the start
        text = re.sub(r'\^[a-zA-Z]\s+', ' ', text) 
        # Substituting multiple spaces with single space
        text = re.sub(r'\s+', ' ', text, flags=re.I)
        # Removing prefixed 'b'
        text = re.sub(r'^b\s+', '', text)
        # Converting to Lowercase
        text = text.lower()
        # Lemmatization
        text = text.split()

        text = [stemmer.lemmatize(word) for word in text if word not in (stop_words)]
        text = ' '.join(text)

        all_text.append(text)
    return all_text

def combine_reviews(data):
    X_1, X_2 = clean_sentences(data["Negative_Review"]), clean_sentences(data["Positive_Review"])
    X = []
    for n in range(len(X_1)):
        X.append(X_1[n] + " " + X_2[n])
    return pd.array(X)

def target_score_to_expression(data):
    y_n = data["Reviewer_Score"].values
    y = []
    for value in y_n:
        if value < 4:
            y.append("negative")
        elif value < 8:
            y.append("neutral")
        else:
            y.append("positive")
        
    return pd.array(y, dtype='object')

def target_score_to_n(data):
    y_n = data["Reviewer_Score"].values
    y = []
    for value in y_n:
        if value < 4:
            y.append([1, 0, 0])
        elif value < 8:
            y.append([0, 1, 0])
        else:
            y.append([0, 0, 1])
               
    return y

def prepare_nlp(data):
    return combine_reviews(data), target_score_to_expression(data)

def prepare_lstm(data):
    return combine_reviews(data), target_score_to_n(data)

def prepare_knn(data):
    return data

def prepare_cnn(data):
    return data

[nltk_data] Downloading package wordnet to /Users/ruben/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/ruben/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
